1-1. CSV 파일 읽기

# 파이썬 기초 과제: 주택 데이터 점검
`1.01.house.price.csv`를 읽고 → 자료형 변환 → 조건에 맞는 주택 선별 → 크기당 가격 함수 → 자료구조 정리 순서로 진행

In [21]:
import pandas as pd

file_name = "/content/sample_data/house.price.csv"

try:
    df_house = pd.read_csv(file_name)
except FileNotFoundError:
    print("파일명과 업로드 여부를 확인하세요.")
else:
    print("첫 번째 행:")
    print(df_house.iloc[0])
    print("전체 데이터 행 수:", len(df_house))
    print("price 자료형:", type(df_house.iloc[0]["price"]))

첫 번째 행:
ID        1.0
size     34.0
year      5.0
price     5.0
Name: 0, dtype: float64
전체 데이터 행 수: 10
price 자료형: <class 'numpy.float64'>


In [22]:
file_name = "no_file.csv"

try:
    df_house = pd.read_csv(file_name)
except FileNotFoundError:
    print("파일명과 업로드 여부를 확인하세요.")
else:
    print("첫 번째 행:")
    print(df_house.iloc[0])
    print("전체 데이터 행 수:", len(df_house))
    print("price 자료형:", type(df_house.iloc[0]["price"]))

파일명과 업로드 여부를 확인하세요.


1-2. 문자열을 숫자로 변환하기

In [23]:
houses = []
excluded_count = 0

for i in range(len(df_house)):
    row = df_house.iloc[i]
    try:
        if pd.isna(row["ID"]) or pd.isna(row["size"]) or pd.isna(row["year"]) or pd.isna(row["price"]):
            raise ValueError("필수 값이 비어 있습니다")
        house = {
            "ID": int(row["ID"]),
            "size": int(row["size"]),
            "year": int(row["year"]),
            "price": float(row["price"]),
        }
        houses.append(house)
    except ValueError as e:
        excluded_count += 1
        print(f"[제외] {i + 1}번째 행: {e}")

print("정상 저장 행 수:", len(houses))
print("제외 행 수:", excluded_count)

정상 저장 행 수: 10
제외 행 수: 0


1-3. 조건에 맞는 주택 찾기

In [24]:
houses_test = houses.copy()

houses_test.append({"ID": 101, "year": 5, "price": 3})
houses_test.append({"ID": 102, "size": 35, "year": 5, "price": None})
houses_test.append({"ID": 103, "size": "오류", "year": 5, "price": 3})
houses_test.append({"ID": 104, "size": 35, "year": 5, "price": 3})

selected = []

for house in houses_test:
    if "size" not in house or "price" not in house:
        print(f"[제외] ID {house['ID']}: size 또는 price 키가 없습니다")
        continue

    if pd.isna(house["size"]) or pd.isna(house["price"]):
        print(f"[제외] ID {house['ID']}: size 또는 price 값이 None(결측값)입니다")
        continue

    try:
        size = float(house["size"])
        price = float(house["price"])
    except (ValueError, TypeError):
        print(f"[제외] ID {house['ID']}: 숫자로 변환할 수 없는 값입니다")
        continue

    if size <= 0 or price < 0:
        print(f"[제외] ID {house['ID']}: size가 0 이하이거나 price가 음수입니다")
        continue

    if size >= 30 and price <= 4:
        selected.append(house)

print("선택된 주택 ID:")
for house in selected:
    print(house["ID"])

print("선택된 주택 수:", len(selected))

[제외] ID 101: size 또는 price 키가 없습니다
[제외] ID 102: size 또는 price 값이 None(결측값)입니다
[제외] ID 103: 숫자로 변환할 수 없는 값입니다
선택된 주택 ID:
3
4
5
9
10
104
선택된 주택 수: 6


1-4. 함수와 예외 처리

In [25]:
def unit_price(price, size):
    try:
        price = float(price)
        size = float(size)

        if size <= 0:
            print("크기 오류: 크기는 0보다 커야 합니다.")
            return None

        if price < 0:
            print("가격 오류: 가격은 음수일 수 없습니다.")
            return None

        return price / size

    except ValueError:
        print("값 오류: 숫자로 변환할 수 없는 문자열입니다.")
        return None

    except TypeError:
        print("자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.")
        return None

In [26]:
for house in houses:
    result = unit_price(house["price"], house["size"])
    if result is not None:
        print(f"주택 ID: {house['ID']}, 크기당 가격: {result:.4f}")
    else:
        print(f"주택 ID: {house['ID']}, 크기당 가격: 계산 불가")

주택 ID: 1, 크기당 가격: 0.1471
주택 ID: 2, 크기당 가격: 0.1000
주택 ID: 3, 크기당 가격: 0.1333
주택 ID: 4, 크기당 가격: 0.0789
주택 ID: 5, 크기당 가격: 0.0750
주택 ID: 6, 크기당 가격: 0.0875
주택 ID: 7, 크기당 가격: 0.0885
주택 ID: 8, 크기당 가격: 0.1000
주택 ID: 9, 크기당 가격: 0.0882
주택 ID: 10, 크기당 가격: 0.0853


In [27]:
print(unit_price(4, 40))
print(unit_price("4", "40"))
print(unit_price(4, 0))
print(unit_price(-4, 40))
print(unit_price("오류", 40))
print(unit_price(None, 40))
print(unit_price(6, 30))

0.1
0.1
크기 오류: 크기는 0보다 커야 합니다.
None
가격 오류: 가격은 음수일 수 없습니다.
None
값 오류: 숫자로 변환할 수 없는 문자열입니다.
None
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
None
0.2


1-5. 목적에 맞는 자료구조 선택하기

In [28]:
# [1] 전체 주택 ID → 리스트
# 이유: ID를 원래 순서대로 저장해야 해서 순서가 유지되는 리스트를 선택했다.
# 세트와의 차이: 세트는 순서가 보장되지 않아서 원래 순서를 지킬 수 없다.
# 판단 근거: 순서 유지가 필요한 작업이라서 리스트가 맞다.
house_ids = []
for house in houses:
    house_ids.append(house["ID"])
print("전체 주택 ID:", house_ids)

# [2] 중복 없는 year → 세트
# 이유: 같은 year가 여러 번 나와도 한 번만 저장해야 해서 중복을 허용하지 않는 세트를 선택했다.
# 리스트와의 차이: 리스트는 중복을 허용해서 같은 year가 여러 번 저장된다.
# 판단 근거: 중복 허용 여부가 핵심이고, year는 순서가 필요 없어서 세트가 맞다.
house_years = set()
for house in houses:
    house_years.add(house["year"])
print("중복 없는 year:", house_years)

# [3] ID로 가격 조회 → 딕셔너리
# 이유: ID(키)로 가격(값)을 바로 찾아야 해서 키로 조회할 수 있는 딕셔너리를 선택했다.
# 리스트와의 차이: 리스트는 위치(인덱스)로만 찾을 수 있어서 ID 3의 가격을 찾으려면 하나씩 확인해야 한다.
# 판단 근거: 키를 통한 조회가 필요한 작업이라서 딕셔너리가 맞다.
house_prices = {}
for house in houses:
    house_prices[house["ID"]] = house["price"]
print("ID 3의 가격:", house_prices.get(3))

# [4] 검색 조건(최소 크기 30, 최대 가격 4) → 튜플
# 이유: 한 번 정한 조건이 중간에 바뀌면 안 되어서 생성 후 변경할 수 없는 튜플을 선택했다.
# 리스트와의 차이: 리스트는 원소를 추가, 삭제, 교체할 수 있어서 조건이 실수로 바뀔 수 있다.
# 판단 근거: 변경 가능 여부가 핵심이고, 추가/삭제/교체가 불가능한 튜플이 맞다.
search_condition = (30, 4)
print("최소 크기:", search_condition[0])
print("최대 가격:", search_condition[1])

전체 주택 ID: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
중복 없는 year: {2, 5, 12, 18, 19, 20, 22}
ID 3의 가격: 4.0
최소 크기: 30
최대 가격: 4
